# Negative-condition reconstruction

Generate literature-guided modification plans from articles, supporting information,
and stored successful syntheses, then enumerate their option combinations.

Plans can include both reported and inferred alternatives.
Cartesian combinations are reconstructed conditions, not independently verified failures.
See `docs/negative_extraction.md` for inputs, output interpretation, and curated corrections.

## Inputs and setup

Install `python -m pip install -e ".[mining,notebook]"` from the repository root. Complete document matching and positive extraction first, then run the cells in order. All paths below are repository-relative.

| Input | Main workflow location | Preparation |
| --- | --- | --- |
| Document manifest, CSV | `results/extraction/document_manifest.csv` | Reuse the manifest from notebook 02, with accessible article and SI paths. |
| Positive records, CSV | `results/extraction/positive/mof_extraction.csv` | Produced by notebook 03; retain `doi`, `article_trial_or_failure`, and `article_trial_or_failure_notes`. |
| Successful syntheses, JSON | `results/extraction/positive/mof_json_store/` | Retain the complete DOI directories produced by positive extraction. |
| Prompts and corrections | `prompts/negative_system.txt`, `prompts/negative_user.txt`, `configs/negative_corrections.json` | Included; selected in `configs/negative_extraction.json`. |

For the included illustrative pair, complete positive extraction with `configs/example_positive_extraction.json`, then load `configs/example_negative_extraction.json` below. The [three-paper input template](../Demo/additional_demo_api_needed/README.md) supports the same sequence with replacement article/SI PDFs.

Input checks and dry runs are local. Enable `RUN_MINING` to generate plans, then `RUN_ENUMERATION` to expand the saved plans. The main workflow writes both under `results/extraction/negative/`; the example uses `results/examples/mining/negative/`. An eligible paper needs a `YES` trial/failure flag; inspect the extracted notes and parent records before enumeration.

Implementation: [negative plans](../src/mofinder/extraction/negative.py) and [enumeration](../src/mofinder/extraction/enumerate_failures.py). See the [source-to-code guide](../docs/source_to_code.md) for the original workflow stages and their corresponding functions.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "configs" / "negative_extraction.json").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from mofinder.extraction.negative import (
    load_config,
    validate_config,
    run_from_config,
    enumerate_from_config,
)

config = load_config(PROJECT_ROOT / "configs" / "negative_extraction.json")


## Inspect inputs

The positive extraction CSV supplies the trial/failure flags and notes. The stored
successful synthesis JSONs provide the indexed parent synthesis records. Configuration
paths and model settings are kept in `configs/negative_extraction.json`.


In [ ]:
validation = validate_config(config)
validation


In [ ]:
if not validation["missing_inputs"]:
    selected_papers = run_from_config(config, dry_run=True)
else:
    print("Complete document matching and positive extraction before negative mining.")


## Generate plans and enumerate combinations

Enable `RUN_MINING` when ready. Enter your API key in the hidden prompt when requested, or set `OPENAI_API_KEY` in your environment first. Each paper's plan stores the exact list of successful syntheses shown to the model.
This list identifies the parent records used during enumeration. Inspect the saved evidence notes before expansion.


In [ ]:
RUN_MINING = False

if RUN_MINING:
    import os
    from getpass import getpass

    if not os.environ.get("OPENAI_API_KEY", "").strip():
        os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ").strip()
    if not os.environ["OPENAI_API_KEY"]:
        raise ValueError("An API key is required for this run.")

    mining_result = run_from_config(config)


In [ ]:
if Path(config["csv_out"]).is_file():
    selected_bases = enumerate_from_config(config, dry_run=True)

RUN_ENUMERATION = False

if RUN_ENUMERATION:
    enumeration_result = enumerate_from_config(config)
